#### Read data from data/raw

In [10]:
import pandas as pd

# Load the consolidated Excel file into a DataFrame
data_2017_2020 = pd.read_excel("../data/raw/nhanes_raw_merged_2017_2020.xlsx")
data_2021_2023 = pd.read_excel("../data/raw/nhanes_raw_merged_2021_2023.xlsx")

In [2]:
print(data_2017_2020.columns)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'BPQ030', 'HIQ011', 'PAD680', 'PAQ670', 'ALQ121',
       'SMQ040', 'SMQ020', 'SLD012', 'SLD013', 'OCQ180', 'LBXSAL', 'LBXSCR',
       'LBXSGL', 'LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW',
       'LBXWBCSI', 'LBDTCSI', 'CICLO'],
      dtype='object')


In [3]:
print(data_2021_2023.columns)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'BPQ030', 'HIQ011', 'PAD680', 'PAD790Q', 'PAD790U',
       'ALQ121', 'SMQ040', 'SMQ020', 'SLD012', 'SLD013', 'OCQ180', 'LBXSAL',
       'LBXSCR', 'LBXSGL', 'LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT', 'LBXMCVSI',
       'LBXRDW', 'LBXWBCSI', 'LBDTCSI', 'CICLO'],
      dtype='object')


## Identifying which columns differ

-> all the columns represent the same variables in both datasets (with the same measurement units), except the ones that refer to moderate physical acitivity: the combination of the columns 'PAD790Q' and 'PAD790U' in 2021-2023 represents the same variable as the column 'PAQ670' in 2017-2020, but they are expressed in different units.

In [23]:
# ---------- 2021-2023: PAD790Q + PAD790U -> vezes por ano ----------
df21 = data_2021_2023.copy()

q = df21["PAD790Q"]
codigos21 = [7777, 9999]                      # Refused / Don't know

# Unidade: apanha a letra mesmo que venha como "W" ou "b'W'"; ausente -> <NA>
u = df21["PAD790U"].astype("string").str.upper().str.extract(r"([DWMY])")[0]
fator = u.map({"D": 365, "W": 52, "M": 12, "Y": 1})

convertido = q * fator
# Onde q é um código, mantém o código; nos restantes, usa o valor convertido
df21["PAMODYR"] = convertido.where(~q.isin(codigos21), q)
df21 = df21.drop(columns=["PAD790Q", "PAD790U"])

#----------------------------

df17 = data_2017_2020.copy()

dias = df17["PAQ670"]
codigos17 = [77, 99]                          # Refused / Don't know

convertido = dias * 52
df17["PAMODYR"] = convertido.where(~dias.isin(codigos17), dias)
df17 = df17.drop(columns=["PAQ670"])

### Now we can finally append the cycles

In [34]:
df_all = pd.concat([df17, df21], ignore_index=True, sort=False)

In [35]:
len(df_all)

27493

## First cleaning layer
-> Remove participants who are not adults (age below 18).

In [43]:
df_filt = df_all.copy()  #to keep it safe

df_filt = df_filt[df_filt["RIDAGEYR"] >= 20]
df_filt.shape

(17041, 59)

## Second cleaning layer 
-> Remove participants with missing values in the variables required to compute `PhenoAge`.

In [44]:
pheno_columns = ['LBXSAL', 'LBXSCR', 'LBXSGL','LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXWBCSI']

In [45]:
df_filt = df_filt.dropna(subset=pheno_columns)
df_filt.shape

(13251, 59)

### Computing PhenoAge (Levine et al., 2018)
-> After computing phenoage discard every variable in `pheno_columns` array.

In [39]:
import numpy as np

def calcular_phenoage(df):
    xb = (
        -19.9067
        - 0.0336 * (df["LBXSAL"] * 10)   #albumin
        + 0.0095 * (df["LBXSCR"] * 88.4)  #creatinine
        + 0.1953 * (df["LBXSGL"] * 0.0555)  #glucose
        + 0.0954 * np.log(df["LBXHSCRP"] / 10)  #C-Reactive Protein
        - 0.0120 * df["LBXLYPCT"]  #Lymphocyte
        + 0.0268 * df["LBXMCVSI"]  #mean cell volume
        + 0.3306 * df["LBXRDW"]
        + 0.0019 * df["LBXSAPSI"]
        + 0.0554 * df["LBXWBCSI"]
        + 0.0804 * df["RIDAGEYR"]
    )
    # M = 1 - exp(-1.51714 * exp(xb) / 0.0076927) ;  Nota: 1.51714 = exp(120*0.0076927) - 1
    # ln(1-M) = -1.51714 * exp(xb) / 0.0076927

    ln_1_menos_m = -1.51714 * np.exp(xb) / 0.0076927

    return 141.50225 + np.log(-0.00553 * ln_1_menos_m) / 0.090165

In [46]:
df_filt["phenoage"] = calcular_phenoage(df_filt)

In [ ]:
df_filt = df_filt.drop(columns=pheno_columns)
print(df_filt.shape)
print()
print(df_filt.columns)

(13251, 51)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'BPQ030', 'HIQ011', 'PAD680', 'ALQ121', 'SMQ040',
       'SMQ020', 'SLD012', 'SLD013', 'OCQ180', 'LBDTCSI', 'CICLO', 'PAMODYR',
       'phenoage'],
      dtype='object')


In [55]:
import os
RAW_PATH = "../data/raw"

def exportar_como_tabela_excel(df: pd.DataFrame, nome_ficheiro: str, nome_folha: str = "Dados") -> None:
    """Exporta um DataFrame formatado como uma Tabela Excel interativa."""
    caminho_completo = os.path.join(RAW_PATH, nome_ficheiro)
    
    with pd.ExcelWriter(caminho_completo, engine="xlsxwriter") as writer:
        df.to_excel(
            writer, sheet_name=nome_folha, startrow=1, header=False, index=False, na_rep="NA"
        )
        workbook = writer.book
        worksheet = writer.sheets[nome_folha]
        (max_row, max_col) = df.shape

        column_settings = [{"header": column} for column in df.columns]

        worksheet.add_table(
            0,
            0,
            max_row,
            max_col - 1,
            {
                "columns": column_settings,
                "style": "Table Style Medium 9",
                "name": "TabelaNHANES_RAW",
            },
        )

        for i, col in enumerate(df.columns):
            largura_maxima = max(df[col].astype(str).map(len).max(), len(str(col))) + 3
            largura_maxima = min(largura_maxima, 50)
            worksheet.set_column(i, i, largura_maxima)

    print(f"Ficheiro '{caminho_completo}' gerado com sucesso!")


exportar_como_tabela_excel(df_filt, "nhanes_2017_2023_pheno.xlsx", nome_folha="NHANES")


Ficheiro '../data/raw\nhanes_2017_2023_pheno.xlsx' gerado com sucesso!


In [50]:
from ydata_profiling import ProfileReport

prof = ProfileReport(df_filt) 

prof.to_file(output_file="Dataset_profiling.html")

Export report to file: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


___
## Dealing with missing values

-> Dietary variables (day 1 and day 2 measurements)

There are three possible scenarios for a given `Dietary` variable measurement:

1. *`Participant X` provided data for both days* (complete data);
2. *`Participant X` provided data for Day 1, but not for Day 2* (partially missing data);
3. *`Participant X` did not provide data for either day* (completely missing data).

> **Note:** If a participant does not provide data for Day 1, there is also no measurement for Day 2, indicating a non-random (monotone) missingness pattern.